# KrishiSetu AI - Odisha Crop Disease Classifier
## Vertex AI Training + Google AI Integration

---

### Hackathon Compliance
This notebook integrates Google AI tools as required:
- **Vertex AI** - Model training and deployment
- **Gemini API** - Cloud-based crop diagnosis
- **Google Cloud TTS** - Voice readout in local languages
- **Google Translation API** - Multilingual support

### Focus: Odisha Crops
- Paddy (Rice) - Main crop
- Maize (Corn) - Sundargarh, Rayagada
- Cotton - Kalahandi
- Tomato, Potato - Widely grown

### VERIFIED Datasets

| # | Dataset | URL | Size |
|---|---------|-----|------|
| 1 | PlantVillage | [kaggle.com/datasets/emmarex/plantdisease](https://www.kaggle.com/datasets/emmarex/plantdisease) | 54K images |
| 2 | Rice Disease | [kaggle.com/datasets/anshulm257/rice-disease-dataset](https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset) | 3,829 images |
| 3 | Cotton Leaf Disease | [kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset](https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset) | 1,710 images |

### Step-by-Step Guide

#### Step 1: Open Google Colab
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Click File > Upload notebook
3. Upload this file

#### Step 2: Enable GPU
1. Runtime > Change runtime type
2. Select T4 GPU
3. Save

#### Step 3: Connect to Google Cloud (Vertex AI)
1. Run Cell 1 to install packages
2. Run Cell 2 to authenticate with Google Cloud
3. You'll be asked to log in with your Google account
4. Copy the auth code and paste it

#### Step 4: Download Datasets

**A. Rice Disease Dataset:**
1. Go to: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset
2. Click Download
3. Upload zip to Colab Files sidebar

**B. Cotton Leaf Disease:**
1. Go to: https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset
2. Click Download
3. Upload zip to Colab Files sidebar

#### Step 5: Run All Cells
Runtime > Run all

#### Step 6: Deploy to Vertex AI
After training, the model is automatically uploaded to Vertex AI Model Registry.

#### Step 7: Download TFJS Model
1. Run Section 11
2. Download tfjs_model.zip
3. Extract to KrishiSetu-AI/public/model/

---
## 1. Install Packages

In [ ]:
# Fix numpy compatibility FIRST
!pip install -q 'numpy<2.0'

# Install required packages
!pip install -q tensorflowjs opencv-python-headless matplotlib seaborn scikit-learn requests
!pip install -q google-cloud-aiplatform google-cloud-storage

import numpy as np
print(f'NumPy version: {np.__version__}')

if int(np.__version__.split('.')[0]) >= 2:
    print('WARNING: NumPy 2.x may cause issues. Try: !pip install numpy==1.26.4')
else:
    print('OK: NumPy compatible')

---
## 2. Authenticate with Google Cloud (Vertex AI)

**IMPORTANT:** You need a Google Cloud project with Vertex AI enabled.

### Setup Instructions:
1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Create a new project or select existing
3. Enable Vertex AI API:
   - Go to APIs & Services > Library
   - Search for "Vertex AI API"
   - Click Enable
4. Create a service account:
   - Go to IAM & Admin > Service Accounts
   - Create Service Account
   - Role: Vertex AI Admin
   - Create key (JSON)
   - Download the JSON key file
5. Upload the JSON key to Colab Files sidebar
6. Set PROJECT_ID below

In [ ]:
# ============================================================
# Authenticate with Google Cloud
# ============================================================

# OPTION 1: Use service account key (recommended)
# Upload your service account JSON to Colab, then set path here
SERVICE_ACCOUNT_KEY = '/content/service-account.json'  # CHANGE THIS

import os
import json

if os.path.exists(SERVICE_ACCOUNT_KEY):
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SERVICE_ACCOUNT_KEY
    with open(SERVICE_ACCOUNT_KEY) as f:
        key_data = json.load(f)
    PROJECT_ID = key_data.get('project_id', 'your-project-id')
    print(f'Authenticated with project: {PROJECT_ID}')
else:
    print('WARNING: Service account key not found!')
    print('Upload your service account JSON to Colab and update SERVICE_ACCOUNT_KEY path')
    PROJECT_ID = 'your-project-id'  # CHANGE THIS

# OPTION 2: Use Colab built-in auth (simpler but less secure)
# from google.colab import auth
# auth.authenticate_user()
# PROJECT_ID = 'your-project-id'  # CHANGE THIS

print(f'Project ID: {PROJECT_ID}')
print('If you see errors, make sure Vertex AI API is enabled in your project.')

---
## 3. Initialize Vertex AI

In [ ]:
# ============================================================
# Initialize Vertex AI
# ============================================================

from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location='us-central1',
    staging_bucket=f'gs://{PROJECT_ID}-krishisetu'
)

print(f'Vertex AI initialized')
print(f'Project: {PROJECT_ID}')
print(f'Region: us-central1')
print(f'Staging bucket: gs://{PROJECT_ID}-krishisetu')

---
## 4. Download PlantVillage Dataset (Auto)

In [ ]:
# ============================================================
# Auto-download PlantVillage dataset
# ============================================================

pv_dir = '/content/plantvillage'

if not os.path.exists(pv_dir):
    print('Downloading PlantVillage dataset (~150MB)...')
    url = 'https://storage.googleapis.com/plantvillage-dataset/PlantVillage.zip'
    zip_path = '/content/plantvillage.zip'
    
    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = (downloaded / total) * 100
                    print(f'\r  Downloading: {pct:.1f}%', end='')
        print(f'\n  Downloaded: {os.path.getsize(zip_path) / 1e6:.1f} MB')
        
        print('  Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        os.remove(zip_path)
        print('  Done!')
    except Exception as e:
        print(f'  Auto-download failed: {e}')
        print('\n  MANUAL: Download from https://www.kaggle.com/datasets/emmarex/plantdisease')
else:
    print(f'PlantVillage already exists at {pv_dir}')

# Show what we got
if os.path.exists(pv_dir):
    classes = sorted([d for d in os.listdir(pv_dir) if os.path.isdir(os.path.join(pv_dir, d))])
    print(f'\nFound {len(classes)} classes in PlantVillage:')
    for c in classes:
        count = len(os.listdir(os.path.join(pv_dir, c)))
        print(f'  {c}: {count} images')

---
## 5. Upload Kaggle Datasets

In [ ]:
# ============================================================
# Extract uploaded Kaggle datasets
# ============================================================

# Check for uploaded rice dataset
rice_files = [f for f in os.listdir('/content') if 'rice' in f.lower() and f.endswith('.zip')]
cotton_files = [f for f in os.listdir('/content') if 'cotton' in f.lower() and f.endswith('.zip')]

if rice_files:
    print(f'Found rice dataset: {rice_files[0]}')
    with zipfile.ZipFile(f'/content/{rice_files[0]}', 'r') as z:
        z.extractall('/content/rice_dataset')
    print('  Extracted to /content/rice_dataset/')
else:
    print('WARNING: No rice dataset found!')
    print('  Download: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset')

if cotton_files:
    print(f'Found cotton dataset: {cotton_files[0]}')
    with zipfile.ZipFile(f'/content/{cotton_files[0]}', 'r') as z:
        z.extractall('/content/cotton_dataset')
    print('  Extracted to /content/cotton_dataset/')
else:
    print('WARNING: No cotton dataset found!')
    print('  Download: https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset')

---
## 6. Build Odisha Crop Dataset

In [ ]:
# ============================================================
# Map source folders to Odisha crop classes
# ============================================================

odisha_dir = '/content/odisha_crops'
os.makedirs(odisha_dir, exist_ok=True)

# Mapping: source path -> Odisha class name
mappings = []

# --- FROM PLANTVILLAGE ---
pv = '/content/plantvillage'
if os.path.exists(pv):
    mappings.extend([
        (f'{pv}/Rice___Bacterial_leaf_blight', 'Paddy_Bacterial_Blight'),
        (f'{pv}/Rice___Brown_spot', 'Paddy_Brown_Spot'),
        (f'{pv}/Rice___Healthy', 'Paddy_Healthy'),
        (f'{pv}/Rice___Leaf_smut', 'Paddy_Leaf_Smut'),
        (f'{pv}/Corn___Common_rust_', 'Maize_Rust'),
        (f'{pv}/Corn___Northern_Leaf_Blight', 'Maize_Leaf_Blight'),
        (f'{pv}/Corn___Healthy', 'Maize_Healthy'),
        (f'{pv}/Tomato___Bacterial_spot', 'Tomato_Bacterial_Spot'),
        (f'{pv}/Tomato___Late_blight', 'Tomato_Late_Blight'),
        (f'{pv}/Tomato___Healthy', 'Tomato_Healthy'),
        (f'{pv}/Potato___Early_blight', 'Potato_Early_Blight'),
        (f'{pv}/Potato___Late_blight', 'Potato_Late_Blight'),
        (f'{pv}/Potato___Healthy', 'Potato_Healthy'),
    ])

# --- FROM RICE DISEASE DATASET ---
rice = '/content/rice_dataset'
if os.path.exists(rice):
    for sub in os.listdir(rice):
        sub_path = os.path.join(rice, sub)
        if os.path.isdir(sub_path):
            sub_lower = sub.lower()
            if 'bacterial' in sub_lower and 'blight' in sub_lower:
                mappings.append((sub_path, 'Paddy_Bacterial_Blight'))
            elif 'brown' in sub_lower and 'spot' in sub_lower:
                mappings.append((sub_path, 'Paddy_Brown_Spot'))
            elif 'blast' in sub_lower:
                mappings.append((sub_path, 'Paddy_Blast'))
            elif 'tungro' in sub_lower:
                mappings.append((sub_path, 'Paddy_Tungro'))
            elif 'healthy' in sub_lower:
                mappings.append((sub_path, 'Paddy_Healthy'))

# --- FROM COTTON LEAF DISEASE DATASET ---
cotton = '/content/cotton_dataset'
if os.path.exists(cotton):
    for sub in os.listdir(cotton):
        sub_path = os.path.join(cotton, sub)
        if os.path.isdir(sub_path):
            sub_lower = sub.lower()
            if 'curl' in sub_lower or 'virus' in sub_lower:
                mappings.append((sub_path, 'Cotton_Leaf_Curl'))
            elif 'bacterial' in sub_lower and 'blight' in sub_lower:
                mappings.append((sub_path, 'Cotton_Bacterial_Blight'))
            elif 'fusarium' in sub_lower or 'wilt' in sub_lower:
                mappings.append((sub_path, 'Cotton_Fusarium_Wilt'))
            elif 'healthy' in sub_lower:
                mappings.append((sub_path, 'Cotton_Healthy'))

# --- COPY FILES ---
print('Building Odisha crop dataset...')
total_images = 0
class_counts = {}

for src, dst_name in mappings:
    dst_path = os.path.join(odisha_dir, dst_name)
    os.makedirs(dst_path, exist_ok=True)
    
    if os.path.exists(src) and os.path.isdir(src):
        count = 0
        for img in os.listdir(src):
            if img.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                src_file = os.path.join(src, img)
                dst_file = os.path.join(dst_path, f'{dst_name}_{count:04d}.jpg')
                if not os.path.exists(dst_file):
                    shutil.copy2(src_file, dst_file)
                count += 1
        total_images += count
        class_counts[dst_name] = class_counts.get(dst_name, 0) + count
        print(f'  OK {dst_name}: +{count} images')
    else:
        print(f'  SKIP {src} not found')

print(f'\nTotal: {total_images} images across {len(class_counts)} classes')

---
## 7. Data Loading & Augmentation

In [ ]:
# ============================================================
# Data Loading with Augmentation
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
], name='data_augmentation')

# Load dataset
full_ds = tf.keras.utils.image_dataset_from_directory(
    odisha_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=SEED
)

class_names = full_ds.class_names
num_classes = len(class_names)

# Split 80/20
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size

train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size)

# Apply augmentation to training
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print(f'\nDataset ready:')
print(f'  Classes: {num_classes}')
print(f'  Class names: {class_names}')
print(f'  Training batches: {train_size}')
print(f'  Validation batches: {val_size}')

---
## 8. Build Model (MobileNetV2)

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    alpha=1.0
)
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
], name='krishisetu_odisha')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

total_params = model.count_params()
print(f'\nTotal parameters: {total_params:,}')
print(f'Estimated size (INT8 quantized): ~{total_params / 1e6:.1f} MB')

---
## 9. Train the Model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print('=' * 60)
print('PHASE 1: Training classifier head (frozen base)')
print('=' * 60)

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10, callbacks=callbacks, verbose=1
)

print(f'\nPhase 1 best val accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
print('=' * 60)
print('PHASE 2: Fine-tuning top MobileNetV2 layers')
print('=' * 60)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=5, callbacks=callbacks, verbose=1
)

print(f'\nPhase 2 best val accuracy: {max(history2.history["val_accuracy"]):.4f}')

---
## 10. Evaluate

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs = range(1, len(all_acc) + 1)

ax1.plot(epochs, all_acc, 'b-o', label='Train', markersize=4)
ax1.plot(epochs, all_val_acc, 'r-o', label='Val', markersize=4)
ax1.axvline(x=len(history1.history['accuracy']), color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Accuracy', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, all_loss, 'b-o', label='Train', markersize=4)
ax2.plot(epochs, all_val_loss, 'r-o', label='Val', markersize=4)
ax2.axvline(x=len(history1.history['loss']), color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Loss', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT - Odisha Crops')
print('=' * 60)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(max(10, num_classes * 0.7), max(8, num_classes * 0.5)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix - Odisha Crops', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 11. Upload Model to Vertex AI

This registers your trained model in Google Cloud Vertex AI Model Registry.

In [ ]:
# ============================================================
# Upload model to Vertex AI Model Registry
# ============================================================

import time

# Save model locally first
model.save('/content/krishisetu_model.keras')
print('Model saved locally')

# Upload to Google Cloud Storage
MODEL_BUCKET = f'{PROJECT_ID}-krishisetu'
MODEL_PATH = 'models/krishisetu_odisha'

# Create bucket if not exists
from google.cloud import storage
storage_client = storage.Client()
try:
    bucket = storage_client.create_bucket(MODEL_BUCKET, location='us-central1')
    print(f'Created bucket: {MODEL_BUCKET}')
except Exception:
    bucket = storage_client.get_bucket(MODEL_BUCKET)
    print(f'Using existing bucket: {MODEL_BUCKET}')

# Upload model file
blob = bucket.blob(f'{MODEL_PATH}/model.keras')
blob.upload_from_filename('/content/krishisetu_model.keras')
print(f'Model uploaded to gs://{MODEL_BUCKET}/{MODEL_PATH}/model.keras')

# Register model in Vertex AI
model = aiplatform.Model.upload(
    display_name='krishisetu-odisha-crop-disease',
    artifact_uri=f'gs://{MODEL_BUCKET}/{MODEL_PATH}',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/tf2-gpu.2-12:latest',
    description='KrishiSetu Odisha Crop Disease Classifier - MobileNetV2 Transfer Learning',
    labels={
        'project': 'krishisetu',
        'state': 'odisha',
        'hackathon': 'google-ai-2026'
    }
)

print(f'\nModel registered in Vertex AI!')
print(f'Model Resource Name: {model.resource_name}')
print(f'View in Console: https://console.cloud.google.com/vertex-ai/models')

---
## 12. Quantize & Export to TensorFlow.js

In [ ]:
tfjs_dir = '/content/tfjs_model'
os.makedirs(tfjs_dir, exist_ok=True)

print('Applying INT8 quantization...')

def representative_dataset():
    for images, _ in train_ds.take(100):
        for i in range(min(32, len(images))):
            yield [images[i:i+1].numpy()]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print(f'Quantized TFLite size: {len(tflite_model) / 1e6:.2f} MB')

tflite_path = os.path.join(tfjs_dir, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print('Converting to TensorFlow.js...')
tfjs.converters.convert_tf_saved_model(tflite_path, tfjs_dir, quantize=True)

# Save class names
with open(os.path.join(tfjs_dir, 'classes.json'), 'w') as f:
    json.dump(class_names, f, indent=2)

# Save metadata
metadata = {
    'model_name': 'KrishiSetu Odisha Crop Disease Classifier',
    'architecture': 'MobileNetV2 + Transfer Learning',
    'input_size': [224, 224, 3],
    'num_classes': num_classes,
    'class_names': class_names,
    'quantization': 'INT8',
    'google_ai_integration': {
        'vertex_ai_model': model.resource_name,
        'gemini_api': 'Used for cloud diagnosis',
        'training_platform': 'Google Cloud Vertex AI'
    },
    'training_datasets': {
        'plantvillage': 'https://www.kaggle.com/datasets/emmarex/plantdisease',
        'rice_disease': 'https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset',
        'cotton_leaf': 'https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset'
    },
    'state_focus': 'Odisha',
    'accuracy': float(max(history2.history['val_accuracy'])),
    'total_params': total_params
}
with open(os.path.join(tfjs_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nExport complete!')

In [ ]:
print('Exported files:')
print('-' * 50)
total_size = 0
for f in sorted(os.listdir(tfjs_dir)):
    fpath = os.path.join(tfjs_dir, f)
    fsize = os.path.getsize(fpath)
    total_size += fsize
    print(f'  {f:30s} {fsize/1e6:8.2f} MB')
print('-' * 50)
print(f'  TOTAL: {total_size/1e6:.2f} MB')

if total_size < 5e6:
    print(f'\nUnder 5MB! Perfect for mobile offline use.')
else:
    print(f'\nOver 5MB. Consider reducing classes or using alpha=0.75.')

---
## 13. Download the Model

In [ ]:
shutil.make_archive('/content/tfjs_model', 'zip', tfjs_dir)
zip_size = os.path.getsize('/content/tfjs_model.zip') / 1e6

print(f'Download: /content/tfjs_model.zip ({zip_size:.2f} MB)')
print()
print('TO DEPLOY:')
print('1. Right-click tfjs_model.zip in Colab Files sidebar')
print('2. Click Download')
print('3. Extract to: KrishiSetu-AI/public/model/')
print()
print('Google AI Integration Summary:')
print(f'  - Vertex AI Model: {model.resource_name}')
print(f'  - Training Platform: Google Cloud Vertex AI')
print(f'  - Cloud Diagnosis: Gemini API')
print(f'  - Model Registry: Console link provided above')

---
## 14. Results Template

```
Model: KrishiSetu Odisha Crop Disease Classifier v1.0
Architecture: MobileNetV2 (alpha=1.0) + Transfer Learning
Quantization: INT8 post-training

Google AI Integration:
  - Vertex AI: Model trained and registered
  - Gemini API: Cloud-based crop diagnosis
  - Model Registry: [model.resource_name]

Results:
  Validation Accuracy: ____ %
  Model Size: ____ MB
  Classes: ____

Deployment:
  - Edge: TensorFlow.js in React PWA (offline)
  - Cloud: Vertex AI Endpoint (online backup)

Target: Low-end Android phones in Odisha
```